# Import Libraries

In [2]:
import pyspark
import pandas as pd
import numpy as np
import math

In [47]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, LongType, IntegerType, FloatType
from pyspark.sql.functions import col, column, trim, explode, array_contains
from pyspark.sql.functions import expr
from pyspark.sql.functions import split
from pyspark.sql import Row

# Create SparkSessions and SparkContext

In [4]:
ss=SparkSession.builder.master("local").appName("Spotify Playlist Generator").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/28 17:34:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
ss.sparkContext.setCheckpointDir("~/scratch")

# Read Data

In [11]:
spotify_DF = ss.read.csv("spotify_dataset.csv", header=True, multiLine=True, escape='"', quote='"', sep=',', inferSchema=True)

# Split Data into Training, Validation, and Testing

In [38]:
train_df, validation_df, test_df = spotify_DF.randomSplit([0.8, 0.1, 0.1], seed=42)

In [39]:
train_df.printSchema()

root
 |-- Artist(s): string (nullable = true)
 |-- song: string (nullable = true)
 |-- text: string (nullable = true)
 |-- Length: string (nullable = true)
 |-- emotion: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Album: string (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- Key: string (nullable = true)
 |-- Tempo: integer (nullable = true)
 |-- Loudness (db): string (nullable = true)
 |-- Time signature: string (nullable = true)
 |-- Explicit: string (nullable = true)
 |-- Popularity: integer (nullable = true)
 |-- Energy: integer (nullable = true)
 |-- Danceability: integer (nullable = true)
 |-- Positiveness: integer (nullable = true)
 |-- Speechiness: integer (nullable = true)
 |-- Liveness: integer (nullable = true)
 |-- Acousticness: integer (nullable = true)
 |-- Instrumentalness: integer (nullable = true)
 |-- Good for Party: integer (nullable = true)
 |-- Good for Work/Study: integer (nullable = true)
 |-- Good for Relaxation/Meditati

In [40]:
train_df.head()

Row(Artist(s)='!!!', song='All My Heroes Are Weirdos', text='Hey ho, there’s an open casting call for heroes And all that showed up was Nero And it’s all blurry but somehow he was hired  Lay low ’cause he’s gone and pissed himself laughing Pissed in the wind and said it was raining Said "don’t worry, it’ll put out the fire"  If I do say so there’s a lotta more story to unfold \'Cause it’ll twist and it’ll turn till it’s been told That the truth is its own supply  I don’t wanna talk about what was Can we please not talk about what is? Why should we talk about what should be When we could talk about what could be?  Hey ho, somewhere a dreamer ain’t so silly after all Sees a crack where others see only wall Yeah all my heroes are weirdos Yeah all my heroes are weirdos Hey ho, there’s an open casting call for heroes And all that showed up was Nero And it’s all blurry but somehow he was hired  Lay low ’cause he’s gone and pissed himself laughing Pissed in the wind and said it was raining Sa

# Sample 10% of the data for exploring data locally

In [41]:
sampled_df = train_df.sample(withReplacement=False, fraction=0.1, seed=42)

In [42]:
sampled_df.count()

44044

# Find the number of genres in dataset

In [43]:
sampled_df_genres = sampled_df.select(explode(split(col("Genre"), ","))).alias("Genre")
sampled_distinct_genres = sampled_df_genres.select(trim(col("col"))).distinct()

In [44]:
sampled_all_genres = [row[0] for row in distinct_genres.collect()]
print(sampled_all_genres)

['electropop', 'folk', 'experimental', 'indie pop', 'post-hardcore', 'pop', 'alternative', 'pop rock', 'math rock', 'new wave', 'k-pop', 'rnb', 'grime', 'ambient', 'chillout', 'christian', 'screamo', 'blues', 'drum and bass', 'dance', 'psychedelic rock', 'cloud rap', 'shoegaze', 'britpop', 'black metal', 'electronic', 'heavy metal', 'dub', 'doom metal', 'trap', 'hip-hop', 'reggaeton', 'techno', 'grunge', 'rap', 'jazz', 'emo rap', 'hip hop', 'death metal', 'trip-hop', 'alt-country', 'country', 'hardcore', 'industrial', 'progressive metal', 'punk rock', 'metalcore', 'chillwave', 'alternative rock', 'dream pop', 'metal', 'soul', 'hard rock', 'psychedelic', 'pop punk', 'progressive rock', 'power metal', 'lo-fi', 'synthpop', 'dancehall', 'indie', 'classic rock', 'electro', 'deathcore', 'indie rock', 'house', 'funk', 'worship', 'dubstep', 'gospel', 'thrash metal', 'acoustic', 'rock', 'post-punk', 'reggae', 'soundtrack', 'emo', 'j-pop', 'punk', 'garage rock', 'disco', 'classical', 'latin', 'n

In [45]:
len(sampled_all_genres)

88

# Turn genre column into lists of genres for each row

In [46]:
sampled_df = sampled_df.withColumn("Genre", split(col("Genre"), ","))

In [50]:
sampled_df_filtered = sampled_df.filter(array_contains(col("Genre"), "comedy"))

In [55]:
sampled_df_filtered.select(["Artist(s)", "song"]).head(20)

[Row(Artist(s)='"Weird Al" Yankovic', song='Dare To Be Stupid'),
 Row(Artist(s)='"Weird Al" Yankovic', song='Everything You Know Is Wrong'),
 Row(Artist(s)='"Weird Al" Yankovic', song='Fun Zone'),
 Row(Artist(s)='"Weird Al" Yankovic', song='George Of The Jungle'),
 Row(Artist(s)='"Weird Al" Yankovic', song='Grapefruit Diet'),
 Row(Artist(s)='"Weird Al" Yankovic', song="Here's Johnny"),
 Row(Artist(s)='"Weird Al" Yankovic', song='I Want a New Duck'),
 Row(Artist(s)='"Weird Al" Yankovic', song="I'll Be Mellow When I'm Dead"),
 Row(Artist(s)='"Weird Al" Yankovic', song='Let the Pun Fit the Crime'),
 Row(Artist(s)='"Weird Al" Yankovic', song='Like a Surgeon'),
 Row(Artist(s)='"Weird Al" Yankovic', song="Livin' In The Fridge"),
 Row(Artist(s)='"Weird Al" Yankovic', song='Mission Statement'),
 Row(Artist(s)='"Weird Al" Yankovic', song='One More Minute'),
 Row(Artist(s)='"Weird Al" Yankovic', song='Party in the C.I.A.'),
 Row(Artist(s)='"Weird Al" Yankovic', song='Pretty Fly For A Rabbi'),
 R

In [53]:
sampled_df_filtered.count()

98

In [ ]:
ss.stop()